# CrewAI: Building Multi-Agent Systems
## Boeing Edition (VS Code)

This notebook provides a comprehensive, practical guide to CrewAI — a framework for orchestrating role-playing autonomous AI agents that work together to complete complex tasks.

All examples use **`gpt-4o-mini`** as the language model and **`text-embedding-3-small`** for embedding operations, served through Boeing's internal `BoeingChatModel` and `BoeingEmbeddings` wrappers (UDAL PAT authentication) instead of calling OpenAI directly.

> **Running locally in VS Code.** Create a `.env` file next to this notebook with: `UDAL_PAT` (required), `SERPER_API_KEY` (optional, needed only for the web-search examples).

> **A note on two integration points.** CrewAI has two different places where a model gets plugged in, and they behave differently once you're not using OpenAI directly:
>
> - **`Agent(llm=...)`** — CrewAI accepts a LangChain-style chat model object here. This notebook passes `chat_client` (`BoeingChatModel`) directly, the same way earlier notebooks in this series did.
> - **Crew `memory`/`embedder` config, and `crewai_tools` RAG search tools like `PDFSearchTool`** — these are powered by `embedchain`/`litellm` under the hood and only understand named providers (`openai`, `azure`, `ollama`, ...). They cannot take a raw Python object. The only way to route them through Boeing is if Boeing's endpoint is **OpenAI-API compatible** — in which case you point the `openai` provider at it via `api_base`. If it isn't, memory and the RAG search tools in this notebook won't work as written; ask your platform team for a compatible URL, or skip those specific cells and rely on the plain agent/task/crew examples.

---

## Table of Contents

1. Installation and Environment Setup
2. Core Concepts: Agent, Task, Crew
3. Agent Properties Deep Dive
4. Task Properties Deep Dive
5. Crew Properties and Process Types
6. Built-in Tools Overview
7. Implementation 1 — Research and Report Writing Crew
8. Implementation 2 — Software Development Crew
9. Implementation 3 — Customer Support Triage Crew
10. Implementation 4 — Financial Analysis Crew with Memory
11. Implementation 5 — RAG-Powered Knowledge Base Crew
12. Advanced: Custom Tools
13. Advanced: Human-in-the-Loop
14. Advanced: Async Execution and Callbacks
15. Best Practices and Common Pitfalls


---
## 1. Installation and Environment Setup

In [ ]:
# ============================================================
# SECTION 0a - ENVIRONMENT SETUP (READ ME FIRST)
# Run these commands in your terminal, NOT inside this cell.
# ============================================================

# ------------------------------------------------------------
# 1) CREATE A VIRTUAL ENVIRONMENT (run once, from your project folder)
# ------------------------------------------------------------
# Windows (PowerShell / cmd):
#     python -m venv venv
#     venv\Scripts\activate
#
# macOS / Linux:
#     python3 -m venv venv
#     source venv/bin/activate
#
# Then select this venv as the Jupyter kernel in VS Code:
#     Command Palette (Ctrl+Shift+P) -> "Python: Select Interpreter" -> ./venv

# ------------------------------------------------------------
# 2) INSTALL PACKAGES
# ------------------------------------------------------------
# pip install crewai crewai-tools python-dotenv
#
# Boeing internal LLM/embeddings wrappers (install from your internal
# package index / artifactory - adjust the source as required by your org):
# pip install boeing-chat-model boeing-embeddings

# ------------------------------------------------------------
# 3) CREATE A .env FILE (same folder as this notebook)
# ------------------------------------------------------------
# UDAL_PAT=your_udal_personal_access_token_here
# SERPER_API_KEY=your_serper_api_key_here          # optional, for web-search examples
# BOEING_CHAT_ENDPOINT=https://your-internal-endpoint       # optional, see Section 0c
# BOEING_EMBEDDING_ENDPOINT=https://your-internal-endpoint  # optional, see Section 0c


In [ ]:
import subprocess, sys

packages = ["crewai", "crewai-tools", "python-dotenv"]

print("[SETUP] Installing dependencies...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + packages, check=True)
print("[SETUP] Dependencies installed.")

# NOTE: BoeingChatModel / BoeingEmbeddings are internal packages.
# Uncomment and adjust the line below to install them from your
# organization's package index if they are not already installed.
# subprocess.run([sys.executable, "-m", "pip", "install", "-q", "boeing-chat-model", "boeing-embeddings"], check=True)


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# ============================================================
# LOAD ENVIRONMENT VARIABLES
# (Local / VS Code friendly - replaces Google Colab Secrets)
# ============================================================

UDAL_PAT = os.getenv("UDAL_PAT")
if not UDAL_PAT:
    raise ValueError(
        "UDAL_PAT not found in .env file.\n"
        "Create a file named '.env' next to this notebook containing:\n"
        "    UDAL_PAT=your_udal_personal_access_token_here"
    )

# Only needed for the web-search examples (SerperDevTool) later in this notebook.
os.environ["SERPER_API_KEY"] = os.getenv("SERPER_API_KEY", "")

# Optional: only needed if Boeing exposes an OpenAI-API-compatible endpoint
# for crewai's memory system / crewai_tools RAG search tools (see note at
# the top of this notebook).
BOEING_CHAT_ENDPOINT      = os.getenv("BOEING_CHAT_ENDPOINT", "")
BOEING_EMBEDDING_ENDPOINT = os.getenv("BOEING_EMBEDDING_ENDPOINT", "")

print("[SETUP] Environment loaded.")
print(f"  UDAL_PAT      : {'set' if UDAL_PAT else 'missing'}")
print(f"  SERPER_API_KEY: {'set' if os.environ['SERPER_API_KEY'] else 'not set (web-search examples will fail)'}")

# ============================================================
# embedchain/litellm-shaped config for crewai memory + crewai_tools
# RAG search tools (PDFSearchTool, etc.)
#
# These subsystems only understand named providers - they cannot accept a
# raw Python object like BoeingEmbeddings/BoeingChatModel. This shape works
# ONLY if BOEING_CHAT_ENDPOINT / BOEING_EMBEDDING_ENDPOINT point to an
# OpenAI-API-compatible URL. If Boeing's endpoint isn't wire-compatible
# with the OpenAI API, skip the memory/RAG cells later in this notebook.
# ============================================================

BOEING_LLM_CONFIG = {
    "provider": "openai",
    "config": {
        "model": "gpt-4o-mini",
        "api_key": UDAL_PAT,
        **({"api_base": BOEING_CHAT_ENDPOINT} if BOEING_CHAT_ENDPOINT else {}),
    },
}

BOEING_EMBEDDER_CONFIG = {
    "provider": "openai",
    "config": {
        "model": "text-embedding-3-small",
        "api_key": UDAL_PAT,
        **({"api_base": BOEING_EMBEDDING_ENDPOINT} if BOEING_EMBEDDING_ENDPOINT else {}),
    },
}

print("[SETUP] BOEING_LLM_CONFIG / BOEING_EMBEDDER_CONFIG ready for crew memory and RAG tools.")


In [ ]:
from boeing_chat_model import BoeingChatModel
from boeing_embeddings import BoeingEmbeddings

# Define the LLM used across all agents in this notebook.
# Passed directly to Agent(llm=...) - CrewAI accepts a LangChain-style
# chat model object here.
chat_client = BoeingChatModel(
    udal_pat=UDAL_PAT,
    model="gpt-4o-mini",
    max_tokens=2048,
    temperature=0.2,           # Lower temperature for more consistent, professional output
    **({"endpoint": BOEING_CHAT_ENDPOINT} if BOEING_CHAT_ENDPOINT else {}),
)

# Define the embedding model - used directly wherever LangChain-style
# embeddings are expected in this notebook.
embedding_client = BoeingEmbeddings(
    udal_pat=UDAL_PAT,
    model="text-embedding-3-small",
    **({"endpoint": BOEING_EMBEDDING_ENDPOINT} if BOEING_EMBEDDING_ENDPOINT else {}),
)

# `llm` is the variable name used by every Agent(...) call throughout this
# notebook, exactly as in the original guide - only its value has changed.
llm = chat_client

print(f"LLM: {chat_client.model}")
print(f"Embeddings: {embedding_client.model}")


---
## 2. Core Concepts: Agent, Task, Crew

CrewAI is built around three primary primitives:

| Primitive | Purpose |
|-----------|-------------------------------------------|
| **Agent** | An autonomous entity with a role, goal, and backstory. It uses tools and an LLM to reason and act. |
| **Task** | A discrete unit of work assigned to one or more agents, with a clear expected output. |
| **Crew** | An orchestrator that manages a collection of agents and tasks, defines the execution process, and aggregates results. |

The typical workflow:
1. Define your agents with specific roles.
2. Define tasks and assign them to agents.
3. Assemble a crew and run it.


---
## 3. Agent Properties Deep Dive

Below is a minimal agent followed by a fully configured agent with explanations for every property.

In [ ]:
from crewai import Agent

minimal_agent = Agent(
    role="Data Analyst",
    goal="Extract actionable insights from raw datasets.",
    backstory=(
        "You are a senior data analyst with ten years of experience "
        "working in the financial services industry."
    ),
    llm=chat_client,
)

print("Minimal agent created:", minimal_agent.role)


In [ ]:
from crewai_tools import SerperDevTool, ScrapeWebsiteTool

# --- Fully Configured Agent ---
fully_configured_agent = Agent(
    # ----------------------------------------------------------------
    # IDENTITY PROPERTIES
    # ----------------------------------------------------------------
    role="Senior Market Research Analyst",
    # role: A short label that defines who this agent is.
    # It influences how the LLM frames its reasoning and responses.

    goal=(
        "Produce accurate, evidence-based market research reports "
        "that help executives make high-confidence investment decisions."
    ),
    # goal: What this agent is trying to achieve. The LLM uses this
    # as an objective to optimize toward throughout task execution.

    backstory=(
        "You spent 15 years as a research analyst at top-tier consulting "
        "firms including McKinsey and BCG. You have deep expertise in "
        "technology, healthcare, and consumer sectors. Your reports are "
        "known for their precision and depth."
    ),
    # backstory: A rich narrative that shapes the agent's persona.
    # The more detailed and realistic, the better the output quality.

    # ----------------------------------------------------------------
    # LLM AND TOOL PROPERTIES
    # ----------------------------------------------------------------
    llm=chat_client,
    # llm: The language model powering this agent (BoeingChatModel).
    # Each agent can have a different LLM instance if needed.

    tools=[SerperDevTool(), ScrapeWebsiteTool()],
    # tools: A list of Tool objects the agent can invoke.
    # Tools give agents the ability to take actions beyond text generation.

    # ----------------------------------------------------------------
    # BEHAVIOR PROPERTIES
    # ----------------------------------------------------------------
    verbose=True,
    # verbose: When True, prints the agent's internal reasoning chain
    # (Thought -> Action -> Observation loop) to stdout. Useful for debugging.

    allow_delegation=False,
    # allow_delegation: When True, this agent can delegate subtasks
    # to other agents in the crew. Set to False to keep it focused.

    max_iter=10,
    # max_iter: Maximum number of reasoning iterations before the agent
    # is forced to produce a final answer. Prevents infinite loops.

    max_rpm=20,
    # max_rpm: Maximum API requests per minute. Useful for rate-limit management.

    memory=True,
    # memory: When True, the agent retains context across task interactions
    # within the same crew run. Powered by the embedding model.

    # ----------------------------------------------------------------
    # OPTIONAL: CUSTOM SYSTEM PROMPT
    # ----------------------------------------------------------------
    # system_template: Override the default system prompt entirely.
    # Use {role}, {goal}, {backstory} as placeholders.
    # system_template="You are {role}. Your goal: {goal}. Background: {backstory}.",
)

print("Fully configured agent created:", fully_configured_agent.role)


### Agent Property Summary

| Property | Type | Required | Description |
|---|---|---|---|
| `role` | str | Yes | The agent's job title / persona label |
| `goal` | str | Yes | What the agent aims to accomplish |
| `backstory` | str | Yes | Rich narrative that shapes behavior |
| `llm` | LLM | No | Language model (defaults to OpenAI GPT-4) |
| `tools` | list | No | Tool objects the agent can call |
| `verbose` | bool | No | Print reasoning steps (default: False) |
| `allow_delegation` | bool | No | Allow task hand-off to other agents |
| `max_iter` | int | No | Max reasoning loops (default: 15) |
| `max_rpm` | int | No | Rate limit for API calls |
| `memory` | bool | No | Enable cross-task memory |
| `system_template` | str | No | Custom system prompt template |

---
## 4. Task Properties Deep Dive

In [ ]:
from crewai import Task
from pydantic import BaseModel
from typing import List

# Define a Pydantic model for structured output
class MarketReport(BaseModel):
    company_name: str
    market_size_usd_billion: float
    key_competitors: List[str]
    growth_rate_percent: float
    recommendation: str

# --- Minimal Task ---
minimal_task = Task(
    description="Summarize the current state of the electric vehicle market.",
    expected_output="A 300-word summary covering market size, key players, and growth trends.",
    agent=minimal_agent,
)

# --- Fully Configured Task ---
research_task = Task(
    # ----------------------------------------------------------------
    # CORE PROPERTIES
    # ----------------------------------------------------------------
    description=(
        "Conduct a comprehensive market analysis for {company_name} "
        "operating in the {industry} sector. "
        "Focus on: (1) total addressable market size, "
        "(2) top 5 competitors, (3) annual growth rate, "
        "(4) a strategic recommendation for market entry."
    ),
    # description: The full instruction for the task. Supports
    # {variable} placeholders that get filled in at crew kickoff.

    expected_output=(
        "A structured JSON report containing: company_name, "
        "market_size_usd_billion, key_competitors (list of 5), "
        "growth_rate_percent, and recommendation."
    ),
    # expected_output: Describes what a successful completion looks like.
    # The LLM uses this to self-evaluate and format its response.

    agent=fully_configured_agent,
    # agent: The agent responsible for this task.

    # ----------------------------------------------------------------
    # STRUCTURED OUTPUT
    # ----------------------------------------------------------------
    output_pydantic=MarketReport,
    # output_pydantic: Force the output to conform to a Pydantic model.
    # CrewAI will validate and parse the output automatically.

    # output_json=MarketReport,  # Alternative: output as a plain dict
    # output_file="report.md",   # Alternative: save output to a file

    # ----------------------------------------------------------------
    # DEPENDENCY PROPERTIES
    # ----------------------------------------------------------------
    # context=[another_task],
    # context: A list of tasks whose outputs are passed as context
    # to this task. Enables data flow between sequential steps.

    # ----------------------------------------------------------------
    # CALLBACK
    # ----------------------------------------------------------------
    # callback=my_callback_function,
    # callback: A Python function called with the task output
    # after the task completes. Useful for logging or downstream actions.

    # ----------------------------------------------------------------
    # HUMAN INPUT
    # ----------------------------------------------------------------
    human_input=False,
    # human_input: When True, the agent pauses and requests human
    # review/approval before finalizing the output.
)

print("Tasks created successfully.")

### Task Property Summary

| Property | Type | Required | Description |
|---|---|---|---|
| `description` | str | Yes | What needs to be done (supports `{placeholders}`) |
| `expected_output` | str | Yes | What a completed result looks like |
| `agent` | Agent | Yes | The responsible agent |
| `context` | list[Task] | No | Upstream tasks whose output feeds this one |
| `output_pydantic` | BaseModel | No | Validate output against a Pydantic model |
| `output_json` | BaseModel | No | Return output as a plain dict |
| `output_file` | str | No | Write output to a file path |
| `callback` | callable | No | Function called on task completion |
| `human_input` | bool | No | Pause for human review before finalizing |

---
## 5. Crew Properties and Process Types

In [ ]:
from crewai import Crew, Process

# Process.sequential  - tasks run one after another in order
# Process.hierarchical - a manager agent orchestrates which agent does what

# --- Sequential Crew (most common) ---
sequential_crew = Crew(
    agents=[minimal_agent],
    tasks=[minimal_task] if "minimal_task" in dir() else [],

    process=Process.sequential,
    # process: Defines the execution strategy.
    # Sequential: task[0] -> task[1] -> task[2] ...
    # Hierarchical: a manager agent delegates to worker agents.

    verbose=True,
    # verbose: Print crew-level orchestration logs.

    memory=False,
    # memory: Enable short-term, long-term, and entity memory
    # across the entire crew. Requires an embedding model.

    embedder=BOEING_EMBEDDER_CONFIG,
    # embedder: Configuration for the embedding model used in memory.
    # See the note at the top of this notebook on why this is shaped as an
    # "openai" provider config rather than passing embedding_client directly.

    max_rpm=30,
    # max_rpm: Crew-level rate limit (applies across all agents).

    # share_crew=False,
    # share_crew: If True, shares crew metadata with CrewAI
    # for performance benchmarking (opt-in).

    # step_callback=my_step_fn,
    # step_callback: Called after every agent action step.

    # task_callback=my_task_fn,
    # task_callback: Called after every task completes.
)

print("Sequential crew assembled.")


In [ ]:
from crewai import Agent, Task, Crew, Process

worker_agent_1 = Agent(
    role="Web Researcher",
    goal="Find accurate information from online sources.",
    backstory="You are a meticulous researcher who always cites sources.",
    llm=chat_client,
    verbose=True,
)

worker_agent_2 = Agent(
    role="Content Writer",
    goal="Produce clear, concise, and well-structured written content.",
    backstory="You are a professional business writer with an economics background.",
    llm=chat_client,
    verbose=True,
)

write_task = Task(
    description="Write a 500-word executive summary on renewable energy trends.",
    expected_output="A professional executive summary suitable for a board presentation.",
    agent=worker_agent_2,
)

hierarchical_crew = Crew(
    agents=[worker_agent_1, worker_agent_2],
    tasks=[write_task],
    process=Process.hierarchical,
    manager_llm=chat_client,
    verbose=True,
)

print("Hierarchical crew assembled.")


---
## 6. Built-in Tools Overview

CrewAI ships with a broad library of tools through the `crewai-tools` package.

In [ ]:
# Overview of key tools available in crewai-tools

tool_catalog = {
    "SerperDevTool": "Google Search via Serper API — good for finding current web results",
    "ScrapeWebsiteTool": "Scrape full text content from any URL",
    "FileReadTool": "Read content from a local file",
    "FileWriterTool": "Write content to a local file",
    "DirectoryReadTool": "List and read files in a directory",
    "PDFSearchTool": "Semantic search over PDF documents using RAG",
    "CSVSearchTool": "Semantic search over CSV files",
    "TXTSearchTool": "Semantic search over plain text files",
    "JSONSearchTool": "Semantic search over JSON documents",
    "DOCXSearchTool": "Semantic search over Word documents",
    "YoutubeChannelSearchTool": "Search a YouTube channel's transcripts",
    "YoutubeVideoSearchTool": "Search a YouTube video's transcript",
    "GithubSearchTool": "Search GitHub repositories and code",
    "CodeDocsSearchTool": "Semantic search over code documentation",
    "WebsiteSearchTool": "RAG-based semantic search over a website",
    "BrowserbaseLoadTool": "Load pages using a headless browser (for JS-heavy sites)",
    "DallETool": "Generate images using DALL-E",
    "VisionTool": "Analyze images with GPT-4 Vision",
}

print(f"{'Tool Name':<30} {'Description'}")
print("-" * 90)
for name, desc in tool_catalog.items():
    print(f"{name:<30} {desc}")

In [ ]:
from crewai.memory import Memory

custom_memory = Memory(
    embedder=BOEING_EMBEDDER_CONFIG,
)


In [ ]:
from crewai import Agent, Task, Crew, Process
from crewai_tools import SerperDevTool, ScrapeWebsiteTool

# --- Agents ---
researcher = Agent(
    role="Senior Investment Research Analyst",
    goal=(
        "Gather comprehensive, factual information about companies "
        "including financial performance, competitive position, and market trends."
    ),
    backstory=(
        "You are a CFA charterholder with 12 years of experience in equity research "
        "at a leading investment bank. You rely only on verifiable data and cite all sources."
    ),
    llm=chat_client,
    tools=[SerperDevTool(), ScrapeWebsiteTool()],
    verbose=True,
    allow_delegation=False,
    max_iter=8,
    memory=True,
)

report_writer = Agent(
    role="Business Report Writer",
    goal=(
        "Convert research into clear, structured, executive-ready investment memos."
    ),
    backstory=(
        "You are a former Bloomberg and Wall Street Journal writer. "
        "You follow the Pyramid Principle and write with clarity and precision."
    ),
    llm=chat_client,
    verbose=True,
    allow_delegation=False,
    max_iter=6,
    memory=True,
)

# --- Tasks ---
research_task = Task(
    description=(
        "Research {company_name} in the {industry} industry.\n\n"
        "Collect:\n"
        "1. Latest revenue and profit figures\n"
        "2. Top 3 competitors with positioning\n"
        "3. Recent news impacting stock price\n"
        "4. Analyst consensus rating\n\n"
        "IMPORTANT: Include source URLs for every fact."
    ),
    expected_output=(
        "Structured bullet-point research brief with clearly labeled sections "
        "and source links for each data point."
    ),
    agent=researcher,
)

writing_task = Task(
    description=(
        "Using the research brief, write a one-page investment memo on {company_name}.\n\n"
        "Structure:\n"
        "- Executive Summary (2 sentences)\n"
        "- Company Overview (1 paragraph)\n"
        "- Financial Highlights (bullets)\n"
        "- Competitive Landscape (1 paragraph)\n"
        "- Risks (3 bullets)\n"
        "- Recommendation (1 paragraph)\n\n"
        "Ensure clarity, conciseness, and professional tone."
    ),
    expected_output=(
        "A polished, executive-ready investment memo in Markdown format."
    ),
    agent=report_writer,
    context=[research_task],
    output_file="investment_memo.md",
)

# --- Crew ---
research_crew = Crew(
    agents=[researcher, report_writer],
    tasks=[research_task, writing_task],
    process=Process.sequential,
    memory=custom_memory,
    verbose=True,
)

# --- Run ---
result = research_crew.kickoff(inputs={
    "company_name": "NVIDIA",
    "industry": "semiconductors and AI computing"
})

print("\n" + "=" * 60)
print("FINAL OUTPUT")
print("=" * 60)
print(result)


In [ ]:
from crewai import Agent, Task, Crew, Process
from crewai_tools import SerperDevTool, ScrapeWebsiteTool

# --- Manager Agent ---
manager = Agent(
    role="Investment Research Manager",
    goal=(
        "Oversee the research and writing process to produce a high-quality "
        "investment memo. Decide task order and delegate effectively."
    ),
    backstory=(
        "You are a senior portfolio manager overseeing analysts and writers. "
        "You break down problems and assign tasks efficiently."
    ),
    llm=chat_client,
    verbose=True,
    allow_delegation=True,   # IMPORTANT
)

# --- Worker Agents ---
researcher = Agent(
    role="Senior Investment Research Analyst",
    goal="Gather comprehensive and factual company data with sources.",
    backstory="CFA with 12 years of equity research experience.",
    llm=chat_client,
    tools=[SerperDevTool(), ScrapeWebsiteTool()],
    verbose=True,
    allow_delegation=False,
)

report_writer = Agent(
    role="Business Report Writer",
    goal="Write clear, structured investment memos.",
    backstory="Former Bloomberg/WSJ writer.",
    llm=chat_client,
    verbose=True,
    allow_delegation=False,
)

# --- Tasks (no strict ordering now) ---
research_task = Task(
    description="Research {company_name} in {industry} with financials, competitors, news, ratings. Include sources.",
    expected_output="Bullet-point research brief with sources.",
    agent=researcher,
)

writing_task = Task(
    description="Write a structured investment memo using research.",
    expected_output="Executive-ready memo in Markdown.",
    agent=report_writer,
)

# --- Crew ---
research_crew = Crew(
    agents=[researcher, report_writer],   # ONLY workers
    tasks=[research_task, writing_task],
    process=Process.hierarchical,
    manager_agent=manager,                # manager here
    memory=custom_memory,
    verbose=True,
)
# --- Run ---
result = await research_crew.kickoff_async(
    inputs={
        "company_name": "NVIDIA",
        "industry": "semiconductors and AI computing",
    }
)

print("\n" + "=" * 60)
print("FINAL OUTPUT")
print("=" * 60)
print(result)


In [ ]:
from crewai import Agent, Task, Crew, Process
from crewai_tools import SerperDevTool, ScrapeWebsiteTool

# =========================
# MANAGER
# =========================
manager = Agent(
    role="Chief Investment Officer (Manager)",
    goal=(
        "Deliver a high-quality investment memo by intelligently delegating tasks. "
        "First ensure research is completed, then validated, then writing, then review."
    ),
    backstory=(
        "You are a CIO managing a team of analysts. You ensure accuracy, clarity, "
        "and investment-grade output."
    ),
    llm=chat_client,
    verbose=True,
    allow_delegation=True,
)

# =========================
# RESEARCHER
# =========================
researcher = Agent(
    role="Equity Research Analyst",
    goal="Collect accurate financials, competitors, news, and analyst ratings with sources.",
    backstory="CFA with deep experience in equity markets.",
    llm=chat_client,
    tools=[SerperDevTool(), ScrapeWebsiteTool()],
    verbose=True,
)

# =========================
# VALIDATOR
# =========================
validator = Agent(
    role="Financial Data Validator",
    goal="Verify all research facts and ensure every claim has a valid source.",
    backstory="Ex-auditor ensuring data integrity and correctness.",
    llm=chat_client,
    verbose=True,
)

# =========================
# WRITER
# =========================
writer = Agent(
    role="Investment Memo Writer",
    goal="Write a clean, structured, executive-ready memo.",
    backstory="Former Bloomberg journalist.",
    llm=chat_client,
    verbose=True,
)

# =========================
# REVIEWER
# =========================
reviewer = Agent(
    role="Senior Investment Reviewer",
    goal="Improve clarity, remove fluff, and ensure professional tone.",
    backstory="Portfolio manager reviewing analyst reports.",
    llm=chat_client,
    verbose=True,
)

# =========================
# TASKS
# =========================
research_task = Task(
    description=(
        "Research {company_name} in {industry}.\n"
        "- Financials\n- Competitors\n- News\n- Analyst ratings\n"
        "Include source URLs."
    ),
    expected_output="Detailed bullet research with sources.",
    agent=researcher,
)

validation_task = Task(
    description=(
        "Validate the research:\n"
        "- Check accuracy\n"
        "- Ensure sources exist\n"
        "- Flag inconsistencies"
    ),
    expected_output="Validated research with corrections if needed.",
    agent=validator,
    context=[research_task],
)

writing_task = Task(
    description=(
        "Write investment memo with:\n"
        "- Executive Summary\n- Financials\n- Competition\n- Risks\n- Recommendation"
    ),
    expected_output="Structured Markdown memo.",
    agent=writer,
    context=[validation_task],
)

review_task = Task(
    description=(
        "Improve the memo:\n"
        "- Make concise\n"
        "- Improve tone\n"
        "- Ensure clarity"
    ),
    expected_output="Final polished memo.",
    agent=reviewer,
    context=[writing_task],
)

# =========================
# CREW (HIERARCHICAL)
# =========================
research_crew = Crew(
    agents=[researcher, validator, writer, reviewer],  # workers only
    tasks=[research_task, validation_task, writing_task, review_task],
    process=Process.hierarchical,
    manager_agent=manager,
    verbose=True,
)

# =========================
# RUN
# =========================
print("\nEXPECTED FLOW:")
print("Manager -> Research -> Validate -> Write -> Review -> Output\n")

result = await research_crew.kickoff_async(
    inputs={
        "company_name": "NVIDIA",
        "industry": "semiconductors and AI computing",
    }
)

print("\n" + "=" * 60)
print("FINAL OUTPUT")
print("=" * 60)
print(result)


In [ ]:
from crewai import Agent, Task, Crew, Process
from crewai_tools import SerperDevTool, ScrapeWebsiteTool

# =========================
# PLANNER AGENT
# =========================
planner = Agent(
    role="Execution Planner",
    goal=(
        "Create a step-by-step execution plan for producing an investment memo. "
        "Clearly define which agent should do what and in what order."
    ),
    backstory="Expert in breaking down complex workflows into clear execution steps.",
    llm=chat_client,
    verbose=True,
)

# =========================
# MANAGER
# =========================
manager = Agent(
    role="Chief Investment Officer",
    goal=(
        "Execute the plan efficiently by delegating tasks to the right agents. "
        "Ensure high-quality output."
    ),
    backstory="Senior decision-maker managing analysts and writers.",
    llm=chat_client,
    verbose=True,
    allow_delegation=True,
)

# =========================
# WORKERS
# =========================
researcher = Agent(
    role="Equity Research Analyst",
    goal="Gather accurate financial and market data with sources.",
    backstory="CFA with deep research expertise.",
    llm=chat_client,
    tools=[SerperDevTool(), ScrapeWebsiteTool()],
    verbose=True,
)

validator = Agent(
    role="Data Validator",
    goal="Verify accuracy and consistency of research.",
    backstory="Audit specialist.",
    llm=chat_client,
    verbose=True,
)

writer = Agent(
    role="Investment Memo Writer",
    goal="Write structured and professional investment memo.",
    backstory="Financial journalist.",
    llm=chat_client,
    verbose=True,
)

reviewer = Agent(
    role="Senior Reviewer",
    goal="Refine and improve clarity and quality.",
    backstory="Portfolio manager.",
    llm=chat_client,
    verbose=True,
)

# =========================
# STEP 1: PLAN TASK
# =========================
planning_task = Task(
    description=(
        "Create a detailed execution plan to analyze {company_name} in {industry}.\n"
        "Include:\n"
        "- Step-by-step workflow\n"
        "- Which agent performs each step\n"
        "- Expected outputs\n"
        "Keep it structured and clear."
    ),
    expected_output="Step-by-step execution plan.",
    agent=planner,
)

# =========================
# STEP 2: EXECUTION TASKS
# =========================
research_task = Task(
    description="Perform research on {company_name} with financials, competitors, news.",
    expected_output="Research with sources.",
    agent=researcher,
)

validation_task = Task(
    description="Validate research for accuracy and completeness.",
    expected_output="Validated research.",
    agent=validator,
)

writing_task = Task(
    description="Write investment memo.",
    expected_output="Structured memo.",
    agent=writer,
)

review_task = Task(
    description="Refine and finalize memo.",
    expected_output="Final polished memo.",
    agent=reviewer,
)

# =========================
# CREW 1 -> PLANNING
# =========================
planning_crew = Crew(
    agents=[planner],
    tasks=[planning_task],
    process=Process.sequential,
    verbose=True,
)

# =========================
# RUN PLANNING FIRST
# =========================
plan = planning_crew.kickoff(inputs={
    "company_name": "NVIDIA",
    "industry": "semiconductors and AI computing"
})

print("\n" + "=" * 60)
print("EXECUTION PLAN (BEFORE RUNNING AGENTS)")
print("=" * 60)
print(plan)

# =========================
# CREW 2 -> AUTONOMOUS EXECUTION
# =========================
execution_crew = Crew(
    agents=[researcher, validator, writer, reviewer],
    tasks=[research_task, validation_task, writing_task, review_task],
    process=Process.hierarchical,
    manager_agent=manager,
    verbose=True,
)

# =========================
# RUN EXECUTION
# =========================
result = execution_crew.kickoff(inputs={
    "company_name": "NVIDIA",
    "industry": "semiconductors and AI computing",
})

print("\n" + "=" * 60)
print("FINAL OUTPUT")
print("=" * 60)
print(result)


---
## 7. Implementation 1 — Research and Report Writing Crew

**Scenario:** An investment firm needs a quick market research report on a company before a decision meeting. Two agents collaborate: one researches, one writes.

---
## 8. Implementation 2 — Software Development Crew

**Scenario:** A startup needs a Python module built. A product manager, software engineer, and QA reviewer collaborate to produce production-ready code.

In [ ]:
from crewai import Agent, Task, Crew, Process

# --- Agents ---
product_manager = Agent(
    role="Technical Product Manager",
    goal="Translate business requirements into precise technical specifications.",
    backstory=(
        "You have 8 years of experience as a PM at companies like Stripe and Twilio. "
        "You write specs that engineers can implement without asking follow-up questions."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

software_engineer = Agent(
    role="Senior Python Engineer",
    goal="Write clean, well-documented, production-quality Python code.",
    backstory=(
        "You are a principal engineer who has worked on backend systems processing "
        "millions of requests per day. You follow PEP 8, write docstrings, and always "
        "include type hints and error handling."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

qa_reviewer = Agent(
    role="QA Engineer",
    goal="Identify bugs, edge cases, and security issues in code before it ships.",
    backstory=(
        "You are a QA lead who has caught critical bugs that saved companies from "
        "production incidents. You are thorough and never approve code without "
        "checking for input validation, error handling, and at least 3 edge cases."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

# --- Tasks ---
spec_task = Task(
    description=(
        "Write a technical specification for a Python function called "
        "`calculate_compound_interest`. "
        "Requirements: accepts principal (float), annual rate (float), "
        "years (int), and compounds_per_year (int, default 12). "
        "Returns final amount (float) rounded to 2 decimal places. "
        "Must handle invalid inputs gracefully."
    ),
    expected_output=(
        "A technical spec document including: function signature, parameter "
        "descriptions, return type, formula to use, and 3 input/output examples."
    ),
    agent=product_manager,
)

coding_task = Task(
    description=(
        "Implement the function described in the specification. "
        "Use type hints, a comprehensive docstring, raise ValueError for invalid inputs, "
        "and include 5 unit tests using pytest within the same file."
    ),
    expected_output=(
        "A complete, runnable Python file with the function implementation "
        "and pytest test suite."
    ),
    agent=software_engineer,
    context=[spec_task],
)

review_task = Task(
    description=(
        "Review the provided Python code for: (1) correctness of the compound interest "
        "formula, (2) completeness of input validation, (3) test coverage gaps, "
        "(4) any security or performance concerns. "
        "Provide a PASS or FAIL verdict with specific line-by-line comments."
    ),
    expected_output=(
        "A code review report with: verdict (PASS/FAIL), a list of issues found "
        "(if any), and the corrected code if changes are needed."
    ),
    agent=qa_reviewer,
    context=[coding_task],
    output_file="code_review_report.md",
)

# --- Crew ---
dev_crew = Crew(
    agents=[product_manager, software_engineer, qa_reviewer],
    tasks=[spec_task, coding_task, review_task],
    process=Process.sequential,
    verbose=True,
)

result = dev_crew.kickoff()

print("\n" + "=" * 60)
print("CODE REVIEW RESULT")
print("=" * 60)
print(result)

---
## 9. Implementation 3 — Customer Support Triage Crew

**Scenario:** An e-commerce company receives customer complaints. Agents classify tickets, draft responses, and escalate high-priority issues.

In [ ]:
from crewai import Agent, Task, Crew, Process
from pydantic import BaseModel
from typing import Literal

class TriageResult(BaseModel):
    ticket_id: str
    category: Literal["billing", "shipping", "product", "technical", "other"]
    priority: Literal["low", "medium", "high", "critical"]
    sentiment: Literal["positive", "neutral", "negative", "angry"]
    requires_escalation: bool
    summary: str

# --- Agents ---
triage_agent = Agent(
    role="Customer Support Triage Specialist",
    goal=(
        "Accurately classify customer tickets by category, priority, and sentiment "
        "so that the right team handles them with the right urgency."
    ),
    backstory=(
        "You managed the support inbox at a high-growth SaaS company for 5 years. "
        "You can instantly recognize whether a ticket is a billing dispute, a technical "
        "bug, or a shipping complaint, and you know exactly what makes a ticket critical."
    ),
    llm=llm,
    verbose=True,
)

response_agent = Agent(
    role="Senior Customer Success Manager",
    goal=(
        "Write empathetic, professional, and solution-focused responses that "
        "resolve customer issues on the first contact whenever possible."
    ),
    backstory=(
        "You have a background in psychology and customer experience. You hold a "
        "CSAT score of 4.9/5.0 across 10,000 tickets. Your responses always "
        "acknowledge the issue, apologize where appropriate, and provide a clear next step."
    ),
    llm=llm,
    verbose=True,
)

# --- Tasks ---
triage_task = Task(
    description=(
        "Analyze the following customer support ticket and classify it:\n\n"
        "Ticket ID: {ticket_id}\n"
        "Customer Message: {customer_message}\n\n"
        "Return structured output with: category, priority (critical if the customer "
        "mentions a chargeback, legal action, or data breach), sentiment, "
        "requires_escalation (True if critical or angry + high), and a 1-sentence summary."
    ),
    expected_output="A structured JSON object matching the TriageResult schema.",
    agent=triage_agent,
    output_pydantic=TriageResult,
)

response_task = Task(
    description=(
        "Using the triage classification, draft a customer-facing response email. "
        "Tone must match sentiment: warmer and more apologetic for negative/angry tickets. "
        "If requires_escalation is True, mention that a senior specialist will follow up "
        "within 2 business hours. Always include a ticket reference number."
    ),
    expected_output=(
        "A complete customer response email with subject line and body. "
        "Professional, empathetic, and under 200 words."
    ),
    agent=response_agent,
    context=[triage_task],
)

# --- Crew ---
support_crew = Crew(
    agents=[triage_agent, response_agent],
    tasks=[triage_task, response_task],
    process=Process.sequential,
    verbose=True,
    tracing=True,
)

# Simulate a customer complaint
result = support_crew.kickoff(inputs={
    "ticket_id": "TKT-20481",
    "customer_message": (
        "I ordered a laptop 3 weeks ago and it still has not arrived. "
        "The tracking link on your website has not updated in 10 days. "
        "This is completely unacceptable. I need this for work and if I don't "
        "receive it by tomorrow I will dispute the charge with my credit card company."
    )
})

print("\n" + "=" * 60)
print("SUPPORT CREW OUTPUT")
print("=" * 60)
print(result)

---
## 10. Implementation 4 — Financial Analysis Crew with Memory

**Scenario:** A multi-turn financial analysis session where agents remember previous findings and build on them across multiple crew runs.

In [ ]:
from crewai import Agent, Task, Crew, Process

# Memory-enabled crew persists knowledge between runs
# using short-term (in-run) and long-term (vector store) memory.

financial_analyst = Agent(
    role="Quantitative Financial Analyst",
    goal=(
        "Analyze financial metrics, identify trends, and compute key ratios "
        "to assess the financial health of companies."
    ),
    backstory=(
        "You hold a PhD in Financial Economics from Wharton and spent 10 years "
        "at a quant hedge fund. You think in numbers, ratios, and distributions."
    ),
    llm=chat_client,
    memory=True,
    verbose=True,
)

strategy_advisor = Agent(
    role="Corporate Strategy Advisor",
    goal=(
        "Translate financial analysis into actionable strategic recommendations "
        "that leadership teams can present to their boards."
    ),
    backstory=(
        "You are a former managing director at a Big Four consulting firm. "
        "You bridge the gap between financial data and strategic action."
    ),
    llm=chat_client,
    memory=True,
    verbose=True,
)

# Provide raw financial data as context in the task description
financial_data = """
Company: Acme Corp
FY2023 Revenue: $4.2B (up 18% YoY)
Gross Margin: 62%
Operating Income: $840M
Net Income: $610M
EPS: $3.82
Debt/Equity: 0.45
Current Ratio: 2.1
Free Cash Flow: $520M
R&D Spend: 14% of revenue
Employee Count: 18,400 (up 6% YoY)
"""

analysis_task = Task(
    description=(
        f"Analyze the following financial data for Acme Corp:\n{financial_data}\n"
        "Compute: P/E ratio context, revenue growth sustainability assessment, "
        "capital efficiency score, and flag any red flags or strengths. "
        "Compare ratios to industry medians for enterprise software companies."
    ),
    expected_output=(
        "A financial analysis report with computed ratios, trend commentary, "
        "3 key strengths, and 2 areas of concern."
    ),
    agent=financial_analyst,
)

strategy_task = Task(
    description=(
        "Based on the financial analysis, provide 3 strategic recommendations "
        "for Acme Corp's leadership team. Each recommendation must include: "
        "the strategic action, rationale grounded in the financial data, "
        "expected outcome, and a risk if not acted upon."
    ),
    expected_output=(
        "Three structured strategic recommendations formatted as a board-ready slide outline."
    ),
    agent=strategy_advisor,
    context=[analysis_task],
)

# Memory-enabled crew - uses text-embedding-3-small (via Boeing) for vector memory
financial_crew = Crew(
    agents=[financial_analyst, strategy_advisor],
    tasks=[analysis_task, strategy_task],
    process=Process.sequential,
    memory=True,
    embedder=BOEING_EMBEDDER_CONFIG,
    verbose=True,
)

result = financial_crew.kickoff()
print(result)


---
## 11. Implementation 5 — RAG-Powered Knowledge Base Crew

**Scenario:** A legal team wants agents to answer questions grounded in their internal policy documents using Retrieval-Augmented Generation (RAG).

In [ ]:
from crewai import Agent, Task, Crew, Process
from crewai_tools import PDFSearchTool, TXTSearchTool

# PDFSearchTool uses an embedding model under the hood to chunk, embed, and
# semantically retrieve relevant passages from the provided document before
# passing them to the agent. See the note at the top of this notebook:
# this config shape only works if Boeing's endpoint is OpenAI-API compatible.

# Assumes you have a PDF file at this path
policy_tool = PDFSearchTool(
    pdf="company_policy.pdf",
    config={
        "llm": BOEING_LLM_CONFIG,
        "embedder": BOEING_EMBEDDER_CONFIG,
    }
)

legal_researcher = Agent(
    role="Corporate Legal Researcher",
    goal=(
        "Answer legal and compliance questions accurately by retrieving "
        "relevant clauses from internal policy documents."
    ),
    backstory=(
        "You are a paralegal at a Fortune 500 company specializing in "
        "corporate compliance and employment law. You always cite the "
        "specific section of the policy document in your answers."
    ),
    llm=chat_client,
    tools=[policy_tool],
    verbose=True,
)

rag_task = Task(
    description=(
        "Answer the following employee question using only the content of "
        "the company policy document. Do not speculate beyond what the "
        "document states. Question: {employee_question}"
    ),
    expected_output=(
        "A direct answer to the question, followed by the exact policy section "
        "that supports the answer, and a note if the policy is silent on the topic."
    ),
    agent=legal_researcher,
)

rag_crew = Crew(
    agents=[legal_researcher],
    tasks=[rag_task],
    process=Process.sequential,
    verbose=True,
)

# Example query
result = rag_crew.kickoff(inputs={
    "employee_question": (
        "Am I entitled to paid parental leave if I have been with the company "
        "for 6 months, and if so, how many weeks?"
    )
})

print(result)


---
## 12. Advanced: Custom Tools

You can build any custom tool by subclassing `BaseTool` or using the `@tool` decorator.

In [ ]:
from crewai.tools import BaseTool, tool
import requests
from pydantic import Field

# --- Method 1: @tool decorator (for simple functions) ---
@tool("Currency Converter")
def currency_converter(amount: float, from_currency: str, to_currency: str) -> str:
    """
    Converts a monetary amount from one currency to another.
    Uses a live exchange rate API.
    Example: currency_converter(100, 'USD', 'EUR')
    """
    # In production, use a real API like frankfurter.app
    # This is a placeholder showing the pattern
    url = f"https://api.frankfurter.app/latest?amount={amount}&from={from_currency}&to={to_currency}"
    try:
        response = requests.get(url, timeout=5)
        data = response.json()
        converted = data["rates"][to_currency]
        return f"{amount} {from_currency} = {converted:.2f} {to_currency}"
    except Exception as e:
        return f"Error fetching exchange rate: {str(e)}"


# --- Method 2: BaseTool subclass (for complex tools with state) ---
class DatabaseLookupTool(BaseTool):
    name: str = "Customer Database Lookup"
    description: str = (
        "Looks up customer information from the internal database "
        "by customer ID. Returns account status, plan, and join date."
    )
    # Simulate an in-memory database
    database: dict = Field(default_factory=lambda: {
        "CUST-001": {"name": "Apex Industries", "plan": "Enterprise", "status": "Active", "since": "2021-03-15"},
        "CUST-002": {"name": "Nova Retail Group", "plan": "Pro", "status": "Past Due", "since": "2022-08-22"},
        "CUST-003": {"name": "Meridian Healthcare", "plan": "Starter", "status": "Active", "since": "2023-11-01"},
    })

    def _run(self, customer_id: str) -> str:
        record = self.database.get(customer_id.upper())
        if record:
            return (
                f"Customer: {record['name']} | Plan: {record['plan']} | "
                f"Status: {record['status']} | Customer since: {record['since']}"
            )
        return f"No customer found with ID: {customer_id}"


# Use the custom tools in an agent
billing_agent = Agent(
    role="Billing Support Specialist",
    goal="Resolve billing disputes by looking up account details and performing currency calculations.",
    backstory="You handle billing inquiries and account status checks for a SaaS platform.",
    llm=llm,
    tools=[DatabaseLookupTool(), currency_converter],
    verbose=True,
)

billing_task = Task(
    description=(
        "Look up customer CUST-002 and determine their account status. "
        "If their account is past due, calculate what their monthly Pro plan fee of "
        "$299 USD would be in EUR and GBP for their international billing team."
    ),
    expected_output=(
        "Account status summary with currency conversions and a recommended action."
    ),
    agent=billing_agent,
)

billing_crew = Crew(
    agents=[billing_agent],
    tasks=[billing_task],
    process=Process.sequential,
    verbose=True,
)

result = billing_crew.kickoff()
print(result)

---
## 13. Advanced: Human-in-the-Loop

In [ ]:
from crewai import Agent, Task, Crew, Process

# When human_input=True on a task, CrewAI will pause execution
# after the agent produces its draft output and prompt the user
# to review and optionally provide correction before finalizing.

legal_drafter = Agent(
    role="Legal Contract Drafter",
    goal="Draft legally sound contract clauses based on provided parameters.",
    backstory=(
        "You are a contract attorney specializing in SaaS and technology agreements. "
        "Your clauses are clear, enforceable, and client-protective."
    ),
    llm=llm,
    verbose=True,
)

draft_task = Task(
    description=(
        "Draft a limitation of liability clause for a SaaS agreement where: "
        "- Liability cap is 12 months of fees paid"
        "- Excludes gross negligence and willful misconduct from the cap"
        "- Mutual limitation (applies to both vendor and customer)"
        "- Governed by Delaware law"
    ),
    expected_output="A formal, numbered legal clause ready for insertion into a contract.",
    agent=legal_drafter,
    human_input=True,   # Agent pauses here for human review before finalizing
)

# Note: Running this cell will pause execution and prompt for input
# Uncomment to run:
legal_crew = Crew(
    agents=[legal_drafter],
    tasks=[draft_task],
    process=Process.sequential,
    verbose=True,
)
result = legal_crew.kickoff()
print(result)

print("Human-in-the-loop task configured. Uncomment the crew kickoff to run interactively.")

---
## 14. Advanced: Async Execution and Callbacks

In [ ]:
from crewai import Agent, Task, Crew, Process
from datetime import datetime

# --- Step Callback: Called after every agent action ---
def step_callback(agent_action):
    timestamp = datetime.now().strftime("%H:%M:%S")
    print(f"[{timestamp}] STEP: {agent_action}")

# --- Task Callback: Called when a task completes ---
def task_callback(task_output):
    timestamp = datetime.now().strftime("%H:%M:%S")
    print(f"[{timestamp}] TASK COMPLETED. Output length: {len(str(task_output))} chars")

# --- Async Kickoff: Non-blocking execution ---
async def run_crew_async():
    summarizer = Agent(
        role="Executive Summarizer",
        goal="Produce concise executive summaries of lengthy documents.",
        backstory="You distill complex content into key decisions and action items.",
        llm=llm,
    )

    summary_task = Task(
        description=(
            "Summarize the following meeting notes into 5 bullet points, "
            "each focused on an action item with an owner and deadline:\n\n"
            "Meeting: Q3 Product Roadmap Review\n"
            "Sarah: We need to ship the new dashboard by end of October or we lose the Acme contract.\n"
            "Tom: The API team is blocked on authentication. They need a decision on OAuth vs SAML by Friday.\n"
            "Sarah: Marketing wants a demo environment set up by September 15 for the conference.\n"
            "James: We agreed to deprecate v1 API on November 1. We need a migration guide published by October 1.\n"
            "Tom: Budget approval for the new infrastructure is needed from Finance before September 30."
        ),
        expected_output="5 action items, each with: task, owner, and deadline.",
        agent=summarizer,
        callback=task_callback,
    )

    crew = Crew(
        agents=[summarizer],
        tasks=[summary_task],
        process=Process.sequential,
        step_callback=step_callback,
        verbose=True,
    )

    # kickoff_async returns a coroutine; await it in an async context
    result = await crew.kickoff_async()
    return result

# In a Jupyter notebook, use await directly
import asyncio
result = await run_crew_async()
print("\nFINAL OUTPUT:")
print(result)

---
## 15. Best Practices and Common Pitfalls

### Best Practices

**Agent Design**
- Give each agent a single, clear responsibility. An agent trying to do too many things produces mediocre output for each.
- Write backstories that are specific and credible. Vague backstories produce generic responses.
- Set `allow_delegation=False` on specialized agents to prevent uncontrolled delegation chains.

**Task Design**
- Use numbered requirements in task descriptions. Agents are more likely to cover every point.
- The `expected_output` field acts as a quality rubric — the more specific, the better the output.
- Use `output_pydantic` when downstream code needs to consume structured data.

**Crew Design**
- Default to `Process.sequential` unless you need dynamic orchestration.
- Enable `memory=True` with `text-embedding-3-small` (via Boeing) when agents need to reference earlier context.
- Use `max_rpm` at the crew level to avoid rate limit errors in large crews.

### Common Pitfalls

| Pitfall | Cause | Fix |
|---|---|---|
| Agent produces off-topic output | Backstory or goal too vague | Make backstory more specific and role-constraining |
| Infinite tool loops | `max_iter` too high | Set `max_iter=8` or lower for most tasks |
| Context not flowing between tasks | Missing `context=[prev_task]` | Explicitly link dependent tasks via `context` |
| Pydantic validation errors | LLM output format inconsistent | Add format examples to `expected_output` |
| Rate limit errors on large crews | Too many simultaneous API calls | Set `max_rpm` at agent and crew level |
| `Agent(llm=...)` works but memory/RAG tools fail | Boeing endpoint isn't OpenAI-API compatible | Confirm `BOEING_CHAT_ENDPOINT`/`BOEING_EMBEDDING_ENDPOINT` with your platform team, or skip memory/RAG cells |